# Deep Reaserch

- Using
    - LangGraph
    - Ollama
    - Search tools (Google Serper, Tavily)


In [1]:
import os
import warnings
from pathlib import Path
from typing import Any

# Standard imports
import numpy as np
import pandas as pd
import polars as pl

# Visualization
# import matplotlib.pyplot as plt

# NumPy settings
np.set_printoptions(precision=4)

# Pandas settings
pd.options.display.max_rows = 1_000
pd.options.display.max_columns = 1_000
pd.options.display.max_colwidth = 600

# Polars settings
pl.Config.set_fmt_str_lengths(1_000)
pl.Config.set_tbl_cols(n=1_000)
pl.Config.set_tbl_rows(n=200)

warnings.filterwarnings("ignore")

# Black code formatter (Optional)
%load_ext lab_black

# auto reload imports
%load_ext autoreload
%autoreload 2

In [2]:
from rich.console import Console
from rich.theme import Theme

custom_theme = Theme(
    {
        "white": "#FFFFFF",  # Bright white
        "info": "#00FF00",  # Bright green
        "warning": "#FFD700",  # Bright gold
        "error": "#FF1493",  # Deep pink
        "success": "#00FFFF",  # Cyan
        "highlight": "#FF4500",  # Orange-red
    }
)
console = Console(theme=custom_theme)


def create_path(path: str | Path) -> None:
    """
    Create parent directories for the given path if they don't exist.

    Parameters
    ----------
    path : str | Path
        The file path for which to create parent directories.
    """
    # Convert to Path object if it's a string
    path_obj: Path = Path(path) if isinstance(path, str) else path

    # Get the parent directory and create it if it doesn't exist
    path_obj.parent.mkdir(parents=True, exist_ok=True)


def go_up_from_current_directory(*, go_up: int = 1) -> None:
    """This is used to up a number of directories.

    Params:
    -------
    go_up: int, default=1
        This indicates the number of times to go back up from the current directory.

    Returns:
    --------
    None
    """
    import sys

    CONST: str = "../"
    NUM: str = CONST * go_up

    # Goto the previous directory
    prev_directory = os.path.join(os.path.dirname(__name__), NUM)
    # Get the 'absolute path' of the previous directory
    abs_path_prev_directory = os.path.abspath(prev_directory)

    # Add the path to the System paths
    sys.path.insert(0, abs_path_prev_directory)
    print(abs_path_prev_directory)

In [3]:
go_up_from_current_directory(go_up=2)

from model_config import LocalModel, RemoteModel  # noqa: E402
from settings import refresh_settings  # noqa: E402

settings = refresh_settings()

/Users/mac/Desktop/Projects/RAG-Tutorials


In [4]:
from langchain_openai import ChatOpenAI

model_str_remote: str = RemoteModel.GPT_OSS_120B
model_str_local: str = LocalModel.MISTRAL_7B_INSTRUCT_V0_3_Q4_0

# Deterministic responses
remote_llm = ChatOpenAI(
    api_key=settings.OPENROUTER_API_KEY.get_secret_value(),  # type: ignore
    base_url=settings.OPENROUTER_URL,
    temperature=0.0,
    model=model_str_remote,
)

local_llm = ChatOpenAI(
    api_key=settings.OLLAMA_API_KEY.get_secret_value(),
    base_url=settings.OLLAMA_URL,
    temperature=0.0,
    model=model_str_local,
)

In [10]:
from langchain_community.utilities import GoogleSerperAPIWrapper

search = GoogleSerperAPIWrapper()
# search.run("What is LangGraph?")

In [ ]:
results = search.results("What is LangGraph?")
console.print(results)

In [5]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool


@tool
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b


llm_with_tools = local_llm.bind_tools([add, multiply])

query: str = "What is the addition of 2 and 7?"
response = llm_with_tools.invoke(query)
console.print(response)

AIMessage(
    content='',
    additional_kwargs={
        'tool_calls': [
            {
                'id': 'call_e2ktqmrw',
                'function': {'arguments': '{"a":2,"b":7}', 'name': 'add'},
                'type': 'function',
                'index': 0
            }
        ],
        'refusal': None
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 80,
            'prompt_tokens': 148,
            'total_tokens': 228,
            'completion_tokens_details': None,
            'prompt_tokens_details': None
        },
        'model_name': 'mistral:7b-instruct-v0.3-q4_0',
        'system_fingerprint': 'fp_ollama',
        'id': 'chatcmpl-451',
        'service_tier': None,
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='run--057a0833-083b-4841-b6d5-c22b69f69e59-0',
    tool_calls=[{'name': 'add', 'args': {'a': 2, 'b': 7}, 'id': 'call_e2ktqmrw', 'type': 'tool_call'}],
    usage_metadata={
        'input_tokens': 148,
        'output_tokens': 80,
        'total_tokens': 228,
        'input_token_details': {},
        'output_token_details': {}
    }
)

In [6]:
for call in response.tool_calls:
    if call["name"] == "add":
        result = add.invoke(call["args"])
    elif call["name"] == "multiply":
        result = multiply.invoke(call["args"])
    console.print(f"The result is: {result}")

The result is: 9

In [7]:
query: str = "What's the product of 2 and 7?"
response = llm_with_tools.invoke(query)

for call in response.tool_calls:
    if call["name"] == "add":
        result = add.invoke(call["args"])
    elif call["name"] == "multiply":
        result = multiply.invoke(call["args"])
        console.print(f"The result is: {result}")

The result is: 14

In [8]:
# search = GoogleSerperAPIWrapper(k=3)


@tool
def google_search(query: str) -> Any:
    """Perform a Google search and return results."""
    search = GoogleSerperAPIWrapper(k=3)
    return search.results(query)


llm_with_tools = local_llm.bind_tools([add, multiply, google_search])

In [17]:
query: str = "Who won the recently concluded Fifa Club World Cup?"
response = llm_with_tools.invoke(query)

for call in response.tool_calls:
    if call["name"] == "add":
        result = add.invoke(call["args"])
    elif call["name"] == "multiply":
        result = multiply.invoke(call["args"])
    elif call["name"] == "google_search":
        result = google_search.invoke(call["args"])
    console.print(f"The result is: {result}")

The result is: {'searchParameters': {'q': 'Recently concluded Fifa Club World Cup winner', 'gl': 'us', 'hl': 'en', 
'type': 'search', 'num': 3, 'engine': 'google'}, 'organic': [{'title': 'FIFA Club World Cup', 'link': 
'https://en.wikipedia.org/wiki/FIFA_Club_World_Cup', 'snippet': 'The current world champions are Chelsea, who 
defeated Paris Saint-Germain 3–0 in the 2025 final. FIFA Club World Cup. The FIFA Club World Cup logo used since 
...', 'position': 1}, {'title': 'Who has won the FIFA Club World Cup? Champions by year', 'link': 
'https://www.espn.com/soccer/story/_/id/45553032/who-won-fifa-club-world-cup-champions-year', 'snippet': 'Real 
Madrid has won the tournament a record five times, claiming victory in 2014, 2016, 2017, 2018 and 2022.', 'date': 
'Jul 13, 2025', 'position': 2}, {'title': "Chelsea's first title wins | FIFA Club World Cup", 'link': 
'https://www.fifa.com/en/tournaments/mens/club-world-cup/usa-2025/articles/chelsea-first-title-wins', 'snippet': 
"Enzo Maresca's side defeated PSG to claim the inaugural FIFA Club World Cup. Chelsea have now won every major 
competition the club has ...", 'date': 'Jul 15, 2025', 'position': 3}], 'credits': 1}

In [18]:
messages = [
    HumanMessage(content=query),
    AIMessage(content=str(result)),
]

res = local_llm.invoke(messages)

In [19]:
console.print(res.content)

The information provided is correct as of July 15, 2025. Chelsea won the recently concluded Fifa Club World Cup in
2025 by defeating Paris Saint-Germain 3–0 in the final.

In [20]:
# Without the use of tools
res = local_llm.invoke(query)
console.print(res.content)

The recently concluded FIFA Club World Cup was won by Bayern Munich. They defeated Tigres UANL from Mexico in the 
final match held on February 11, 2020. This was Bayern's first-ever title in this competition.

# Using LangGraph

In [21]:
from typing import Annotated, TypedDict

from langgraph.graph.message import add_messages


class State(TypedDict):
    messages: Annotated[list[Any], add_messages]
    name: str
    date: str

In [ ]:
from IPython.display import Image, display
from langchain.chat_models import init_chat_model
from langchain_core.messages import ToolMessage
from langchain_core.tools import InjectedToolCallId, tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, interrupt


@tool
def human_assistance(name: str, date: str, tool_call_id: Annotated[str, InjectedToolCallId]) -> str:
    hman_response = interrupt(
        {
            "question": "Is this correct?",
            "name": name,
            "date": date,
        }
    )
    # If the info is correct, update the state
    if hman_response.get()